# BTC 1D — Experimento exp11_multitask_dir

Objetivo principal: predicción multiobjetivo **OHLCV**.

Objetivo auxiliar: clasificación **dirección del cierre** (`up/down`) para reforzar señal direccional sin cambiar el target principal.


In [1]:
import sys, subprocess, importlib
packages = ["pandas","numpy","matplotlib","scikit-learn","torch","joblib","tqdm","mplfinance"]
for p in packages:
    try:
        importlib.import_module(p if p != "scikit-learn" else "sklearn")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", p])

import json
import math
import random
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplfinance as mpf
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import joblib
from tqdm.auto import tqdm


/opt/anaconda3/envs/proyecto_ar/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_repo_root():
    start = pathlib.Path(".").resolve()
    for p in [start] + list(start.parents):
        if (p / "experimentos" / "data" / "btc_binance_ohlcv_1d.csv").exists():
            return p
    raise FileNotFoundError(f"No se encontró el repo raíz desde: {start}")

repo_root = find_repo_root()

csv_candidates = [repo_root / "experimentos" / "data" / "btc_binance_ohlcv_1d.csv"]
csv_path = next((p for p in csv_candidates if p.exists()), csv_candidates[0])
if not csv_path.exists():
    raise FileNotFoundError(f"No existe el CSV. Probado: {csv_candidates}")

exp_root = repo_root / "experimentos"
models_root = exp_root / "modelos"
preds_root = exp_root / "predicciones"
figures_root = exp_root / "figuras"

for p in [models_root, preds_root, figures_root]:
    p.mkdir(parents=True, exist_ok=True)

csv_path


PosixPath('/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/experimentos/data/btc_binance_ohlcv_1d.csv')

In [3]:
df = pd.read_csv(csv_path)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values("timestamp").reset_index(drop=True)

df["range_hl"] = (df["high"] - df["low"]).astype(np.float32)
df["body_co"] = (df["close"] - df["open"]).astype(np.float32)
df["abs_log_return"] = df["log_return"].abs().astype(np.float32)
df["log_volume"] = np.log1p(df["volume"]).astype(np.float32)
df["day_of_week"] = df["timestamp"].dt.dayofweek.astype(np.int16)

feature_cols = [
    "open","high","low","close","volume",
    "log_close","log_return",
    "range_hl","body_co","abs_log_return","log_volume","day_of_week",
    "sma_8","ema_8","sma_20","ema_20",
    "sma_100","ema_100","sma_200","ema_200",
    "volatility_7"
]

target_cols = ["open","high","low","close","volume"]

df = df.dropna(subset=feature_cols).reset_index(drop=True)
df.shape


(2910, 22)

## Arquitectura

Backbone: Transformer encoder-only sobre tokens temporales (ventana).

Cabezas:
- Regresión: 5 dimensiones (OHLCV) en espacio relativo logarítmico.
- Clasificación: logits `up/down` del cierre.


In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class MultiTaskTransformer(nn.Module):
    def __init__(self, n_features, out_dim, d_model=64, n_heads=4, n_layers=2, dropout=0.1, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=512)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.regressor = nn.Linear(d_model, out_dim)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)
        h = x[:, -1, :]
        y_reg = self.regressor(h)
        logits = self.classifier(h)
        return y_reg, logits

class SeqDatasetMulti(Dataset):
    def __init__(self, X, y_reg, y_dir):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_reg = torch.tensor(y_reg, dtype=torch.float32)
        self.y_dir = torch.tensor(y_dir, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_reg[idx], self.y_dir[idx]


In [5]:
def mae_1d(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse_1d(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def directional_acc_1d(y_true, y_pred):
    return float(np.mean(np.sign(y_true) == np.sign(y_pred)))

def metrics_log_return(y_true, y_pred):
    return {"mae": mae_1d(y_true, y_pred), "rmse": rmse_1d(y_true, y_pred), "directional_acc": directional_acc_1d(y_true, y_pred)}

def metrics_ohlcv(y_true, y_pred, cols):
    err = y_true - y_pred
    mae_vals = np.mean(np.abs(err), axis=0)
    rmse_vals = np.sqrt(np.mean(err ** 2, axis=0))
    out = {}
    for i, c in enumerate(cols):
        out[f"mae_{c}"] = float(mae_vals[i])
        out[f"rmse_{c}"] = float(rmse_vals[i])
    out["mae_mean"] = float(np.mean(mae_vals))
    out["rmse_mean"] = float(np.mean(rmse_vals))
    return out


## Secuencias (window → horizon)


In [6]:
def build_sequences(features, target_abs, timestamps, close_series, volume_series, window, horizon):
    X, y_abs, y_dates, close_t, volume_t = [], [], [], [], []
    last_index = len(features) - horizon
    for idx in range(window - 1, last_index):
        X.append(features[idx - window + 1: idx + 1])
        y_abs.append(target_abs[idx + horizon])
        y_dates.append(timestamps[idx + horizon])
        close_t.append(close_series[idx])
        volume_t.append(volume_series[idx])
    return np.array(X), np.array(y_abs), np.array(y_dates), np.array(close_t), np.array(volume_t)


## Configuración del experimento


In [7]:
cfg = {
  "run_tag": "exp11_multitask_dir",
  "window": 7,
  "horizon": 1,
  "d_model": 64,
  "n_heads": 4,
  "n_layers": 2,
  "dropout": 0.1,
  "lr": 1e-3,
  "batch_size": 64,
  "epochs": 300,
  "patience": 20,
  "min_epochs": 60,
  "loss_weights": [1.0, 1.0, 1.0, 2.0, 0.5],
  "lambda_dir": 0.2,
  "lr_patience": 5,
  "lr_factor": 0.5,
  "reset_patience_on_lr_drop": True,
  "min_delta": 1e-4
}


In [8]:
run_tag = cfg["run_tag"]
window = int(cfg.get("window", 7))
horizon = int(cfg.get("horizon", 1))

batch_size = int(cfg.get("batch_size", 64))
epochs = int(cfg.get("epochs", 100))
patience = int(cfg.get("patience", 10))
min_epochs = int(cfg.get("min_epochs", 20))
min_delta = float(cfg.get("min_delta", 1e-4))
reset_patience_on_lr_drop = bool(cfg.get("reset_patience_on_lr_drop", True))
lr_patience = int(cfg.get("lr_patience", 5))
lr_factor = float(cfg.get("lr_factor", 0.5))

d_model = int(cfg.get("d_model", 64))
n_heads = int(cfg.get("n_heads", 4))
n_layers = int(cfg.get("n_layers", 2))
dropout = float(cfg.get("dropout", 0.1))
lr = float(cfg.get("lr", 1e-3))

loss_weights = cfg.get("loss_weights", [1.0, 1.0, 1.0, 2.0, 0.5])
lambda_dir = float(cfg.get("lambda_dir", 0.2))

run_dirs = {
    "models": models_root / run_tag,
    "predictions": preds_root / run_tag,
    "figures": figures_root / run_tag,
}
for p in run_dirs.values():
    p.mkdir(parents=True, exist_ok=True)

model_path = run_dirs["models"] / "transformer_btc_1d.pt"
scaler_path = run_dirs["models"] / "scaler_btc_1d.joblib"
y_scaler_path = run_dirs["models"] / "scaler_y_btc_1d.joblib"
config_path = run_dirs["models"] / "config_transformer_btc_1d.json"
metrics_path = run_dirs["models"] / "metrics_transformer_btc_1d.json"
preds_csv_path = run_dirs["predictions"] / "btc_1d_transformer_test_predictions.csv"

print("Run:", run_tag)
print("HParams:", {"window": window, "horizon": horizon, "d_model": d_model, "n_heads": n_heads, "n_layers": n_layers, "dropout": dropout, "lr": lr, "lambda_dir": lambda_dir})
print("Train:", {"batch_size": batch_size, "epochs": epochs, "patience": patience, "min_epochs": min_epochs})
run_dirs


Run: exp11_multitask_dir
HParams: {'window': 7, 'horizon': 1, 'd_model': 64, 'n_heads': 4, 'n_layers': 2, 'dropout': 0.1, 'lr': 0.001, 'lambda_dir': 0.2}
Train: {'batch_size': 64, 'epochs': 300, 'patience': 20, 'min_epochs': 60}


{'models': PosixPath('/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/experimentos/modelos/exp11_multitask_dir'),
 'predictions': PosixPath('/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/experimentos/predicciones/exp11_multitask_dir'),
 'figures': PosixPath('/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/experimentos/figuras/exp11_multitask_dir')}

In [9]:
features = df[feature_cols].values.astype(np.float32)
target_abs = df[target_cols].values.astype(np.float32)
timestamps = df["timestamp"].values
close_series = df["close"].values.astype(np.float32)
volume_series = df["volume"].values.astype(np.float32)

X_all, y_all_abs, y_dates_all, close_t_all, volume_t_all = build_sequences(
    features, target_abs, timestamps, close_series, volume_series, window=window, horizon=horizon
)

eps = 1e-6
base_all = np.column_stack([close_t_all, close_t_all, close_t_all, close_t_all, volume_t_all]).astype(np.float32)
y_all_rel = np.log((y_all_abs + eps) / (base_all + eps)).astype(np.float32)

close_idx = target_cols.index("close")
y_dir_all = (y_all_abs[:, close_idx] > close_t_all).astype(np.int64)

n_total = len(X_all)
train_size = int(n_total * 0.7)
val_size = int(n_total * 0.15)

X_train_raw = X_all[:train_size]
y_train_abs = y_all_abs[:train_size]
y_train_rel = y_all_rel[:train_size]
y_train_dir = y_dir_all[:train_size]

X_val_raw = X_all[train_size:train_size + val_size]
y_val_rel = y_all_rel[train_size:train_size + val_size]
y_val_dir = y_dir_all[train_size:train_size + val_size]

X_test_raw = X_all[train_size + val_size:]
y_test_abs = y_all_abs[train_size + val_size:]
y_test_rel = y_all_rel[train_size + val_size:]
y_test_dir = y_dir_all[train_size + val_size:]
y_dates_test = y_dates_all[train_size + val_size:]
close_t_test = close_t_all[train_size + val_size:]
volume_t_test = volume_t_all[train_size + val_size:]

{"n_total": n_total, "train": len(X_train_raw), "val": len(X_val_raw), "test": len(X_test_raw)}


{'n_total': 2903, 'train': 2032, 'val': 435, 'test': 436}

In [10]:
scaler = StandardScaler()
scaler.fit(X_train_raw.reshape(-1, X_train_raw.shape[-1]))

def scale_X(X):
    X_2d = X.reshape(-1, X.shape[-1])
    return scaler.transform(X_2d).reshape(X.shape)

X_train = scale_X(X_train_raw)
X_val = scale_X(X_val_raw)
X_test = scale_X(X_test_raw)

y_scaler = StandardScaler()
y_scaler.fit(y_train_rel)
y_train = y_scaler.transform(y_train_rel)
y_val = y_scaler.transform(y_val_rel)
y_test = y_scaler.transform(y_test_rel)

joblib.dump(scaler, scaler_path)
joblib.dump(y_scaler, y_scaler_path)

train_loader = DataLoader(SeqDatasetMulti(X_train, y_train, y_train_dir), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(SeqDatasetMulti(X_val, y_val, y_val_dir), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(SeqDatasetMulti(X_test, y_test, y_test_dir), batch_size=batch_size, shuffle=False)

{"X_train": X_train.shape, "y_train": y_train.shape, "y_dir_rate": float(y_train_dir.mean())}


{'X_train': (2032, 7, 21),
 'y_train': (2032, 5),
 'y_dir_rate': 0.5073818897637795}

In [11]:
model = MultiTaskTransformer(
    n_features=X_train.shape[-1],
    out_dim=len(target_cols),
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    dropout=dropout,
).to(device)

criterion_reg = nn.SmoothL1Loss(reduction="none")
lw = torch.tensor(loss_weights, dtype=torch.float32, device=device)
criterion_dir = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=lr_factor, patience=lr_patience)

sum(p.numel() for p in model.parameters())


564167

In [12]:
best_val = float("inf")
best_state = None
train_losses, val_losses = [], []
pat = 0
last_lr = optimizer.param_groups[0]["lr"]

for epoch in range(1, epochs + 1):
    model.train()
    total_train = 0.0
    n_train = 0
    for xb, yb_reg, yb_dir in train_loader:
        xb = xb.to(device)
        yb_reg = yb_reg.to(device)
        yb_dir = yb_dir.to(device)

        optimizer.zero_grad()
        pred_reg, logits = model(xb)
        reg_loss = criterion_reg(pred_reg, yb_reg)
        reg_loss = (reg_loss * lw).mean()
        dir_loss = criterion_dir(logits, yb_dir)
        loss = reg_loss + lambda_dir * dir_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_train += float(loss.detach().cpu().item()) * xb.size(0)
        n_train += xb.size(0)

    train_loss = total_train / max(1, n_train)

    model.eval()
    total_val = 0.0
    n_val = 0
    with torch.no_grad():
        for xb, yb_reg, yb_dir in val_loader:
            xb = xb.to(device)
            yb_reg = yb_reg.to(device)
            yb_dir = yb_dir.to(device)
            pred_reg, logits = model(xb)
            reg_loss = criterion_reg(pred_reg, yb_reg)
            reg_loss = (reg_loss * lw).mean()
            dir_loss = criterion_dir(logits, yb_dir)
            loss = reg_loss + lambda_dir * dir_loss
            total_val += float(loss.detach().cpu().item()) * xb.size(0)
            n_val += xb.size(0)

    val_loss = total_val / max(1, n_val)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    scheduler.step(val_loss)
    lr_now = optimizer.param_groups[0]["lr"]
    lr_dropped = lr_now < last_lr - 1e-15
    if lr_dropped and reset_patience_on_lr_drop:
        pat = 0
    last_lr = lr_now

    improved = (best_val - val_loss) > min_delta
    if improved:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        pat = 0
    else:
        if epoch >= min_epochs:
            pat += 1

    mark = "*" if improved else ""
    lr_mark = " lr↓" if lr_dropped else ""
    print(f"Epoch {epoch:03d} | train={train_loss:.6f} | val={val_loss:.6f} | lr={lr_now:.6g} | best_val={best_val:.6f} | patience={pat}/{patience}{lr_mark} {mark}")

    if epoch >= min_epochs and pat >= patience:
        break

if best_state is not None:
    model.load_state_dict(best_state)

torch.save(model.state_dict(), model_path)

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

fig_loss, ax = plt.subplots(figsize=(8,4))
ax.plot(train_losses, label="train")
ax.plot(val_losses, label="val")
ax.set_title("Pérdida")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
fig_loss.tight_layout()
fig_path_loss = run_dirs["figures"] / "btc_transformer_train_val_loss.png"
fig_loss.savefig(fig_path_loss, dpi=300, bbox_inches="tight")
plt.close(fig_loss)

fig_path_loss


Epoch 001 | train=0.538381 | val=0.380223 | lr=0.001 | best_val=0.380223 | patience=0/20 *
Epoch 002 | train=0.490190 | val=0.377366 | lr=0.001 | best_val=0.377366 | patience=0/20 *
Epoch 003 | train=0.482097 | val=0.380328 | lr=0.001 | best_val=0.377366 | patience=0/20 
Epoch 004 | train=0.479599 | val=0.370456 | lr=0.001 | best_val=0.370456 | patience=0/20 *
Epoch 005 | train=0.472176 | val=0.368431 | lr=0.001 | best_val=0.368431 | patience=0/20 *
Epoch 006 | train=0.468552 | val=0.360933 | lr=0.001 | best_val=0.360933 | patience=0/20 *
Epoch 007 | train=0.465547 | val=0.351476 | lr=0.001 | best_val=0.351476 | patience=0/20 *
Epoch 008 | train=0.464782 | val=0.356639 | lr=0.001 | best_val=0.351476 | patience=0/20 
Epoch 009 | train=0.464515 | val=0.359325 | lr=0.001 | best_val=0.351476 | patience=0/20 
Epoch 010 | train=0.465374 | val=0.354354 | lr=0.001 | best_val=0.351476 | patience=0/20 
Epoch 011 | train=0.459573 | val=0.362742 | lr=0.001 | best_val=0.351476 | patience=0/20 
Epoc

PosixPath('/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/experimentos/figuras/exp11_multitask_dir/btc_transformer_train_val_loss.png')

In [13]:
model.eval()
preds_reg = []
preds_logits = []
with torch.no_grad():
    for xb, _, _ in test_loader:
        xb = xb.to(device)
        y_reg, logits = model(xb)
        preds_reg.append(y_reg.cpu().numpy())
        preds_logits.append(logits.cpu().numpy())

y_pred_scaled = np.concatenate(preds_reg)
y_pred_rel = y_scaler.inverse_transform(y_pred_scaled)

base_test = np.column_stack([close_t_test, close_t_test, close_t_test, close_t_test, volume_t_test]).astype(np.float32)
y_pred = (base_test * np.exp(y_pred_rel)).astype(np.float32)
y_pred[:, 1] = np.maximum.reduce([y_pred[:, 1], y_pred[:, 0], y_pred[:, 3]])
y_pred[:, 2] = np.minimum.reduce([y_pred[:, 2], y_pred[:, 0], y_pred[:, 3]])
y_pred[:, 4] = np.maximum(y_pred[:, 4], 0.0)

logits_all = np.concatenate(preds_logits)
y_pred_dir = logits_all.argmax(axis=1).astype(np.int64)

{"y_pred": y_pred.shape, "dir_acc": float((y_pred_dir == y_test_dir).mean())}


{'y_pred': (436, 5), 'dir_acc': 0.48623853211009177}

In [14]:
model_metrics_ohlcv = metrics_ohlcv(y_test_abs, y_pred, target_cols)
model_metrics_logret = metrics_log_return(y_test_rel[:, close_idx], y_pred_rel[:, close_idx])
model_metrics_dir = {"dir_close_acc": float((y_pred_dir == y_test_dir).mean())}

ohlcv_idx = [feature_cols.index(c) for c in target_cols]
y_pred_persist = X_test_raw[:, -1, ohlcv_idx]

X_train_lr = X_train.reshape(X_train.shape[0], -1)
X_test_lr = X_test.reshape(X_test.shape[0], -1)
lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_abs)
y_pred_lr = lr_model.predict(X_test_lr)

baseline_metrics_ohlcv = {
    "test_ohlcv_persistence": metrics_ohlcv(y_test_abs, y_pred_persist, target_cols),
    "test_ohlcv_linear": metrics_ohlcv(y_test_abs, y_pred_lr, target_cols),
}

y_true_logret = y_test_rel[:, close_idx]
y_pred_logret_zero = np.zeros_like(y_true_logret)
log_return_idx = feature_cols.index("log_return")
y_pred_logret_persist = X_test_raw[:, -1, log_return_idx]

lr_model_logret = LinearRegression()
lr_model_logret.fit(X_train_lr, y_train_rel[:, close_idx])
y_pred_logret_lr = lr_model_logret.predict(X_test_lr)

baseline_metrics_logret = {
    "test_logret_zero": metrics_log_return(y_true_logret, y_pred_logret_zero),
    "test_logret_persistence": metrics_log_return(y_true_logret, y_pred_logret_persist),
    "test_logret_linear": metrics_log_return(y_true_logret, y_pred_logret_lr),
}

all_metrics = {
    "test_model_ohlcv": model_metrics_ohlcv,
    "test_model_logret_diag": model_metrics_logret,
    "test_model_dir_aux": model_metrics_dir,
    "test_baselines_ohlcv": baseline_metrics_ohlcv,
    "test_baselines_logret_diag": baseline_metrics_logret,
}

with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)

pd.DataFrame({
    "timestamp": pd.to_datetime(y_dates_test),
    **{f"y_true_{c}": y_test_abs[:, i] for i, c in enumerate(target_cols)},
    **{f"y_pred_{c}": y_pred[:, i] for i, c in enumerate(target_cols)},
}).to_csv(preds_csv_path, index=False)

all_metrics["test_model_ohlcv"], all_metrics["test_model_dir_aux"]


({'mae_open': 0.5036374926567078,
  'rmse_open': 0.591881513595581,
  'mae_high': 1053.2591552734375,
  'rmse_high': 1418.2607421875,
  'mae_low': 1301.941650390625,
  'rmse_low': 1820.795166015625,
  'mae_close': 1584.6561279296875,
  'rmse_close': 2184.454833984375,
  'mae_volume': 8010.474609375,
  'rmse_volume': 11588.9951171875,
  'mae_mean': 2390.167236328125,
  'rmse_mean': 3402.61962890625},
 {'dir_close_acc': 0.48623853211009177})

In [15]:
n_plot = 250
close_true = y_test_abs[-n_plot:, close_idx]
close_pred = y_pred[-n_plot:, close_idx]
t_plot = pd.to_datetime(y_dates_test[-n_plot:])

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(t_plot, close_true, label="close_real")
ax.plot(t_plot, close_pred, label="close_pred")
ax.set_title("Close — test (últimos puntos)")
ax.legend()
fig.tight_layout()
fig_path = run_dirs["figures"] / "btc_transformer_test_series.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.close(fig)
fig_path


PosixPath('/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/experimentos/figuras/exp11_multitask_dir/btc_transformer_test_series.png')

In [16]:
week = 7
week_actual = y_test_abs[-week:]
week_pred = y_pred[-week:]
week_dates = pd.to_datetime(y_dates_test[-week:])

week_actual_ohlcv = pd.DataFrame(week_actual, columns=target_cols)
week_actual_ohlcv.index = week_dates
week_pred_ohlcv = pd.DataFrame(week_pred, columns=target_cols)
week_pred_ohlcv.index = week_dates

fig_path_week_real = run_dirs["figures"] / "btc_transformer_last_week_actual_candles.png"
mpf.plot(week_actual_ohlcv, type="candle", volume=True, style="charles", savefig=dict(fname=str(fig_path_week_real), dpi=300, bbox_inches="tight"))

fig_path_week_pred = run_dirs["figures"] / "btc_transformer_last_week_pred_candles.png"
mpf.plot(week_pred_ohlcv, type="candle", volume=True, style="charles", savefig=dict(fname=str(fig_path_week_pred), dpi=300, bbox_inches="tight"))

[fig_path_week_real, fig_path_week_pred]


[PosixPath('/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/experimentos/figuras/exp11_multitask_dir/btc_transformer_last_week_actual_candles.png'),
 PosixPath('/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/experimentos/figuras/exp11_multitask_dir/btc_transformer_last_week_pred_candles.png')]